In [2]:
%%javascript
MathJax.Hub.Config({
    TeX: { equationNumbers: { autoNumber: "AMS" } }
});

<IPython.core.display.Javascript object>

$$
\renewcommand{\x}{\mathbf{x}};
\renewcommand{\y}{\mathbf{y}};
\renewcommand{\d}[1]{\dot{#1}};  
\newcommand{\f}[1]{\mathbf{#1}};
$$
## Reactive SINDy: Sparse Learning of Reaction Kinetics
At any given time $t$, it's desired to model the concentration of $S$ species with concentrations,
$$
\begin{align}
    \mathbf{x}(t) = \begin{bmatrix} x_1(t) \\ x_2(t) \\ \vdots \\ x_S(t) \end{bmatrix} \in \mathbb{R}^S
\end{align}
$$
where $x_i(t)$ is the concentration of species $i$ at time $t$. The dynamics of each species is subject to the law of mass action, which states that the rate of change of the concentration of a species is proportional to the product of the concentrations of the reactants. The dynamics of species $s$ at time $t$ is given by the following equation,
$$
\begin{equation}
    \frac{dx_s(t)}{dt} = f_s(\mathbf{x}(t)) = \sum_{i} \beta_{s,0}^{(i)} + \sum_{i,j} \beta_{s,1}^{(i,j)} x_i(t) + \sum_{i,j,k} \beta_{s,2}^{(i,j,k)} x_i(t) x_j(t) + \ldots + \sum_{i_1,i_2,\ldots,i_k} \beta_{s,k}^{(i_1,i_2,\ldots,i_k)} x_{i_1}(t) x_{i_2}(t) \ldots x_{i_k}(t)
    \tag{2}
\end{equation}
$$
where $\beta_{s,k}^{(i_1,i_2,\ldots,i_k)}$ are the constants belonging to the $k$-th order reactions. The first term is the zeroth order reaction, the second term is the first order reaction, and so on. These constants incorporate all the reaction rates associated with the $k$-th order reaction. For example, for the reaction system,
$$
\begin{align}
    s_1 \xrightarrow{\xi_1} s_2 \\
    s_2 \xrightarrow{\xi_2} s_3 \\
\end{align}
$$
The dynamics of the first species is given by,
$$
\begin{align}
    \frac{dx_1(t)}{dt} = \beta_{1,1}^{(1)} x_1(t) = -(\xi_1 + \xi_2) x_1(t) \\
\end{align}
.
$$
Since the dynamics of a species is linear in terms of the rate parameters, the equation \eqref{eq:mass_action} can be written in a more compact form. Consider the following ansatz for reaction $r$,
$$
\begin{align}
    \mathbf{y}_r(\mathbf{x}(t)) = \begin{bmatrix} y_{r,1}(\mathbf{x}(t)) \\ y_{r,2}(\mathbf{x}(t)) \\ \vdots \\ y_{r,S}(\mathbf{x}(t)) \end{bmatrix}, \quad r = 1,2,\ldots,R
\end{align}
$$
such that the dynamics of species $s$ can be expressed as a linear combination of the functions $\mathbf{y}_r(\mathbf{x}(t))$ as follows,
$$
\begin{align}
    \frac{dx_s(t)}{dt} = \sum_{r=1}^R \xi _r y_r(\mathbf{x}(t)), \quad s = 1,2,\ldots,S
\end{align}
$$

in the example above, the ansatz is given by,
$$
\begin{align}
    \mathbf{y}_1(\mathbf{x}(t)) = \begin{bmatrix} - x_1(t) && x_1(t) && 0 \end{bmatrix}^T, \quad \mathbf{y}_2(\mathbf{x}(t)) = \begin{bmatrix} -x_1(t) && 0 && x_1(t) \end{bmatrix}^T 
\end{align}
$$
where the first ansatz is a zeroth order reaction and the second ansatz is a first order reaction. The dynamics of the species can be expressed as follows,
$$ 
\begin{align}
    \begin{bmatrix} \d{x_1}(t) \\ \d{x_2}(t) \\ \d{x_3}(t) \end{bmatrix} = \begin{bmatrix} \y _1(\x(t)) && \y _2(\x(t)) \end{bmatrix} \begin{bmatrix} \xi_1 \\ \xi_2 \end{bmatrix} = 
        \begin{bmatrix} 
            - x_1(t) && -x_1(t) \\
            x_1(t) && 0 \\
            0 && x_1(t) \\
        \end{bmatrix} \begin{bmatrix} \xi_1 \\ \xi_2 \end{bmatrix}
        = \begin{bmatrix} -(\xi_1 + \xi_2) x_1(t) \\ \xi_1 x_1(t) \\ \xi_2 x_1(t) \end{bmatrix}  
\end{align}
$$
Given concetration measurements of the species at time $t$, for $t = 1,2,\ldots,T$, as follows,
$$
\begin{align}
    \mathbf{X} = \begin{bmatrix} \mathbf{x}(t_1) && \mathbf{x}(t_2) && \ldots && \mathbf{x}(t_T) \end{bmatrix} \in \mathbb{R}^{S \times T}
\end{align}
$$

evaluating the ansatz $\mathbf{y}_r(\mathbf{x}(t))$ at the time points $t_1, t_2, \ldots, t_T$ yields the following library of functions,
$$
\begin{align}
    \theta _r(\mathbf{X}) = \begin{bmatrix} \mathbf{y}_r(\mathbf{x}(t_1))^T \\ \mathbf{y}_r(\mathbf{x}(t_2))^T \\ \vdots \\ \mathbf{y}_r(\mathbf{x}(t_T))^T \end{bmatrix} \in \mathbb{R}^{S \times T}, \quad r = 1,2,\ldots,R
\end{align}
$$
Concatenating the library of functions for all the reactions yields the following tensor,
$$
\begin{align}
    \Theta(\mathbf{X}) = \begin{bmatrix} \theta_1(\mathbf{X}) && \theta_2(\mathbf{X}) && \ldots && \theta_R(\mathbf{X}) \end{bmatrix} \in \mathbb{R}^{S \times T \times R}
\end{align}
$$
where $\Theta(\mathbf{X})$ is a 3D tensor. The observed dynamics of the species for all time points can be expressed as follows,
$$
\begin{align}
    \d{\mathbf{X}} = \Theta(\mathbf{X}) \mathbf{\xi} = \sum_{r=1}^R \theta_r(\mathbf{X}) \xi_r
\end{align}
$$
The goal of reactive SINDy is to learn the reaction rates $\f{\xi} = (\xi_1, \xi_2, \ldots, \xi_R)^T$ from the observed dynamics $\d{\mathbf{X}}$ and states $\mathbf{X}$. 

---
header-includes:
  - \usepackage[ruled,vlined,linesnumbered]{algorithm2e}
---
# Algorithm 1
Just a sample algorithmn
\begin{algorithm}[H]
\DontPrintSemicolon
\SetAlgoLined
\KwResult{Write here the result}
\SetKwInOut{Input}{Input}\SetKwInOut{Output}{Output}
\Input{Write here the input}
\Output{Write here the output}
\BlankLine
\While{While condition}{
    instructions\;
    \eIf{condition}{
        instructions1\;
        instructions2\;
    }{
        instructions3\;
    }
}
\caption{While loop with If/Else condition}
\end{algorithm} 

In [4]:
pip install sphinx-proof


   ---------------------------------------- 0.0/587.4 kB ? eta -:--:--
   ---------------- ----------------------- 235.5/587.4 kB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 587.4/587.4 kB 7.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   ------ --------------------------------- 0.6/3.6 MB 13.1 MB/s eta 0:00:01
   ---------------- ----------------------- 1.5/3.6 MB 15.4 MB/s eta 0:00:01
   --------------------------- ------------ 2.5/3.6 MB 17.6 MB/s eta 0:00:01
   ------------------------------------- -- 3.4/3.6 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 3.6/3.6 MB 16.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/434.0 kB ? eta -:--:--
   --------------------------------------- 434.0/434.0 kB 13.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.6 MB 26.4 MB/s eta 0:00:01
   ------------ --


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
